## Training from scratch on only 325 samples (no data augmentation for least representative samples)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Define the base path to your data in Google Drive
base_path = '/content/drive/MyDrive/data_minecapmf/data/processed'

# Check if the path exists
if os.path.exists(base_path):
    print(f"Found data directory: {base_path}")

    # List the contents (folders) within the processed directory
    categories = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    print(f"Found {len(categories)} categories: {categories}")

else:
    print(f"Error: Data directory not found at {base_path}. Please check the path.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found data directory: /content/drive/MyDrive/data_minecapmf/data/processed
Found 7 categories: ['step', 'ambient', 'combat', 'mob', 'damage', 'token_cache', 'runs']


In [ ]:
import librosa

for category in categories:
    category_path = os.path.join(base_path, category)
    print(f"\nProcessing category: {category}")
    # List sound files in this category
    sound_files = [f for f in os.listdir(category_path) if f.endswith(('.wav', '.flac', '.mp3'))]
    for sound_file in sound_files[:2]: # Process first 2 files as an example
        file_path = os.path.join(category_path, sound_file)
        print(f"  Loading {sound_file}...")
        y, sr = librosa.load(file_path, sr=None)
        print(f"    Loaded audio with shape {y.shape} and sample rate {sr}")


Processing category: step
  Loading scaffold_walk (1).wav...
    Loaded audio with shape (64000,) and sample rate 16000
  Loading grass_walk.wav...
    Loaded audio with shape (64000,) and sample rate 16000

Processing category: ambient

Processing category: combat
  Loading blaze_encounter_slow.wav...
    Loaded audio with shape (64000,) and sample rate 16000
  Loading blaze_encounter.wav...
    Loaded audio with shape (64000,) and sample rate 16000

Processing category: mob

Processing category: damage
  Loading hit_seq_fast.wav...
    Loaded audio with shape (64000,) and sample rate 16000
  Loading fallsmall_seq.wav...
    Loaded audio with shape (64000,) and sample rate 16000

Processing category: token_cache

Processing category: runs
  Loading sanity_reconstruction.wav...
    Loaded audio with shape (96000,) and sample rate 24000
  Loading gen_zombie_cave.wav...
    Loaded audio with shape (96000,) and sample rate 24000


In [ ]:
# --- Imports and reproducibility ---
import os
import re
import json
import math
import random
from dataclasses import dataclass
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", DEVICE)
if DEVICE == "cpu":
    print("Warning: CPU only. Training will be slow. Use CUDA for practical runs.")

torch: 2.10.0+cu128
device: cuda


In [ ]:
from pathlib import Path
import json

CAPTIONS_PATH = Path('/content/drive/MyDrive/data_minecapmf/data/processed/_captions.json')
# The base_path variable was defined in an earlier cell
AUDIO_ROOT = Path(base_path)

captions = json.loads(CAPTIONS_PATH.read_text(encoding="utf-8"))

records = []
for file_name, caption in sorted(captions.items()):
    wav_path = AUDIO_ROOT / file_name
    if wav_path.exists() and file_name.endswith(".wav"):
        records.append({"file_name": file_name, "caption": caption})

print("Records:", len(records))
print("Example:", records[0])


Records: 325
Example: {'file_name': 'ambient/cave/cave1.wav', 'caption': 'minecraft cave ambience sound effect'}


In [ ]:
# --- Build event keys and categories ---
variant_re = re.compile(r"_(fast|slow|tgap|wgap)$")

def leaf_category(file_name: str) -> str:
    parts = file_name.split("/")
    if len(parts) >= 3 and parts[0] in {"ambient", "mob"}:
        return "/".join(parts[:2])
    return parts[0]

def event_stem(file_name: str) -> str:
    stem = Path(file_name).stem
    return variant_re.sub("", stem)

def event_key(file_name: str) -> str:
    return f"{leaf_category(file_name)}/{event_stem(file_name)}"

for r in records:
    r["leaf_category"] = leaf_category(r["file_name"])
    r["event_key"] = event_key(r["file_name"])

event_counts = Counter(r["event_key"] for r in records)
print("Distinct events:", len(event_counts))
print("Event size distribution:", Counter(event_counts.values()))
print("Events with <3 variants:", sum(v < 3 for v in event_counts.values()))

Distinct events: 120
Event size distribution: Counter({3: 48, 2: 41, 5: 17, 1: 14})
Events with <3 variants: 55


## Step 1: Coverage-Aware Train/Val/Test Split

Default policy in this notebook is **low-data friendly**:
- Put **all original clips** in `train` (so the model sees every available real sample).
- Build `val` and `test` as deterministic **virtual augmented views** per event.

This guarantees:
- every sound event appears in train, val, and test,
- train keeps maximum data (important for 325-clip regime),
- split is fully deterministic and reproducible.

You can switch to a stricter disjoint policy by setting `KEEP_ALL_ORIGINALS_IN_TRAIN = False` in the next cell.

In [ ]:
from collections import defaultdict
import random
import pandas as pd

SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}


def add_row(out_rows, rec, split):
    out_rows.append({
        "file_name": rec["file_name"],
        "source_file_name": rec["file_name"],
        "caption": rec["caption"],
        "leaf_category": rec["leaf_category"],
        "event_key": rec["event_key"],
        "split": split,
        "virtual_aug": "none",
        "is_virtual": 0,
    })


def build_real_split(records, seed=42):
    rng = random.Random(seed)

    grouped = defaultdict(list)
    for r in records:
        grouped[r["event_key"]].append(r)

    out = []

    for ek in sorted(grouped):
        items = grouped[ek].copy()
        rng.shuffle(items)
        n = len(items)

        if n == 1:
            add_row(out, items[0], "train")

        elif n == 2:
            add_row(out, items[0], "train")
            add_row(out, items[1], "val")

        elif n == 3:
            add_row(out, items[0], "train")
            add_row(out, items[1], "val")
            add_row(out, items[2], "test")

        else:
            n_train = max(2, round(n * 0.7))
            n_val = max(1, round(n * 0.15))
            n_test = n - n_train - n_val

            if n_test < 1:
                n_test = 1
                n_train -= 1

            idx = 0
            for _ in range(n_train):
                add_row(out, items[idx], "train")
                idx += 1
            for _ in range(n_val):
                add_row(out, items[idx], "val")
                idx += 1
            for _ in range(n_test):
                add_row(out, items[idx], "test")
                idx += 1

    df = pd.DataFrame(out)

    df = df.sort_values(
        ["event_key", "split", "file_name"]
    ).reset_index(drop=True)

    df["token_relpath"] = df["file_name"].str.replace("/", "__").str.replace(".wav", "", regex=False) + "__none.npy"
    df["sample_id"] = [f"s{i:05d}" for i in range(len(df))]

    return df

In [ ]:
split_df = build_real_split(records, seed=42)

In [ ]:
# --- Split diagnostics ---
print("Rows per split:")
print(split_df["split"].value_counts())

print("\nOriginal vs virtual rows:")
print(split_df["is_virtual"].value_counts())

print("\nOriginal rows per split:")
print(split_df[split_df["is_virtual"] == 0]["split"].value_counts())

# event coverage check
coverage = split_df.groupby("event_key")["split"].apply(lambda x: set(x.tolist()))
missing = {k: sorted({"train", "val", "test"} - v) for k, v in coverage.items() if {"train", "val", "test"} - v}
print("\nEvents missing any split:", len(missing))

# unique underlying files per split
uniq_by_split = split_df.groupby("split")["file_name"].nunique()
print("\nUnique base files per split:")
print(uniq_by_split)

# category distribution
print("\nLeaf category counts by split:")
print(pd.crosstab(split_df["leaf_category"], split_df["split"]))

Rows per split:
split
train    154
val      106
test      65
Name: count, dtype: int64

Original vs virtual rows:
is_virtual
0    325
Name: count, dtype: int64

Original rows per split:
split
train    154
val      106
test      65
Name: count, dtype: int64

Events missing any split: 55

Unique base files per split:
split
test      65
train    154
val      106
Name: file_name, dtype: int64

Leaf category counts by split:
split               test  train  val
leaf_category                       
ambient/cave           0     23   23
ambient/underwater     0      7    7
ambient/weather        0     11   11
combat                 8      8    8
damage                 3      5    3
mob/blaze              4      8    4
mob/creeper            2      4    2
mob/endermen           9     17    9
mob/ghast              7     11    7
mob/skeleton           5     10    5
mob/spider             3      6    3
mob/zombie            13     22   13
step                  11     22   11


In [ ]:
pip install encodec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 41.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for encodec: filename=encodec-0.1.1-py3-none-any.whl size=45759 sha256=f43c876c523ba9064902aeaa6fed631fcbef8ce90dff799459a5b70627946d00
  Stored in directory: /root/.cache/pip/wheels/b8/eb/9f/e13610cc46ab39d3199fbabebd1c3e142d44b679526e0f228a
Successfully built encodec


## Step 2: EnCodec Tokenization at 1.5 kbps

- Model: `encodec_model_24khz`
- Bandwidth: `1.5 kbps`
- Expected codebooks: `2`
- For 4s clips: about `75 frames/s * 4 * 2 = 600` interleaved tokens

In [ ]:
# --- Load EnCodec and set tokenization constants ---
from encodec import EncodecModel

ENCODEC_BANDWIDTH = 1.5
DURATION_S = 4.0

encodec_model = EncodecModel.encodec_model_24khz()
encodec_model.set_target_bandwidth(ENCODEC_BANDWIDTH)
encodec_model.to(DEVICE)
encodec_model.eval()

ENCODEC_SR = encodec_model.sample_rate
ENCODEC_CHANNELS = encodec_model.channels
FRAME_RATE = encodec_model.frame_rate

N_CODEBOOKS = 2  # at 1.5 kbps for 24k model
CODEBOOK_SIZE = 1024
TOKEN_SEQ_LEN = int(DURATION_S * FRAME_RATE * N_CODEBOOKS)

print("ENCODEC_SR:", ENCODEC_SR)
print("FRAME_RATE:", FRAME_RATE)
print("N_CODEBOOKS:", N_CODEBOOKS)
print("TOKEN_SEQ_LEN:", TOKEN_SEQ_LEN)
print("CODEBOOK_SIZE:", CODEBOOK_SIZE)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:01<00:00, 65.8MB/s]


ENCODEC_SR: 24000
FRAME_RATE: 75
N_CODEBOOKS: 2
TOKEN_SEQ_LEN: 600
CODEBOOK_SIZE: 1024


In [ ]:
# --- Audio loading, virtual augmentation, and token helpers ---
def load_audio_mono(path: Path):
    wav, sr = sf.read(str(path), always_2d=False)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)
    return wav, int(sr)


def apply_virtual_aug(wav: np.ndarray, aug: str):
    if aug == "none":
        return wav
    if aug.startswith("speed_"):
        rate = float(aug.split("_")[1])
        if len(wav) < 16:
            return wav
        return librosa.effects.time_stretch(wav, rate=rate)
    raise ValueError(f"Unknown virtual augmentation: {aug}")


def standardize_audio(wav: np.ndarray, sr: int, target_sr: int, duration_s: float):
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)

    target_len = int(duration_s * target_sr)
    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    else:
        wav = wav[:target_len]

    wav = np.clip(wav, -1.0, 1.0)
    return wav.astype(np.float32)


def interleave_codes(codes: torch.Tensor) -> torch.Tensor:
    # codes: [K, T] -> [T*K], order [c1_t1, c2_t1, c1_t2, c2_t2, ...]
    return codes.transpose(0, 1).reshape(-1)


def deinterleave_tokens(tokens: torch.Tensor, n_codebooks: int) -> torch.Tensor:
    # tokens: [T*K] -> [K, T]
    t = tokens.numel() // n_codebooks
    return tokens.view(t, n_codebooks).transpose(0, 1)


def pad_or_trim_1d(x: torch.Tensor, target_len: int, pad_value: int = 0):
    if x.numel() < target_len:
        pad = torch.full((target_len - x.numel(),), pad_value, dtype=x.dtype)
        return torch.cat([x, pad], dim=0)
    return x[:target_len]


def tokenize_row_with_encodec(row):
    wav_path = AUDIO_ROOT / row["file_name"]
    wav, sr = load_audio_mono(wav_path)
    wav = apply_virtual_aug(wav, row["virtual_aug"])
    wav = standardize_audio(wav, sr=sr, target_sr=ENCODEC_SR, duration_s=DURATION_S)

    # EnCodec expects [B, C, T]
    x = torch.from_numpy(wav).to(torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        encoded_frames = encodec_model.encode(x)

    # For 24k model segment=None, we generally have one frame.
    # robustly concatenate in case multiple frames are returned
    codes = torch.cat([ef[0] for ef in encoded_frames], dim=-1)  # [B, K, T]
    codes = codes[0].cpu().to(torch.long)  # [K, T]

    if codes.shape[0] != N_CODEBOOKS:
        raise RuntimeError(f"Expected {N_CODEBOOKS} codebooks at 1.5kbps, got {codes.shape[0]}")

    tokens = interleave_codes(codes)  # [T*K]
    tokens = pad_or_trim_1d(tokens, TOKEN_SEQ_LEN, pad_value=0)
    return tokens


def decode_tokens_with_encodec(tokens_1d: torch.Tensor):
    tokens_1d = tokens_1d.to(torch.long)
    tokens_1d = pad_or_trim_1d(tokens_1d, TOKEN_SEQ_LEN, pad_value=0)
    codes = deinterleave_tokens(tokens_1d, N_CODEBOOKS).unsqueeze(0).to(DEVICE)  # [1, K, T]
    encoded_frames = [(codes, None)]
    with torch.no_grad():
        wav = encodec_model.decode(encoded_frames)
    # wav: [B, C, T]
    return wav[0, 0].detach().cpu().numpy()


In [ ]:
# --- Build token cache for split rows ---
from pathlib import Path

TOKEN_CACHE_DIR = Path(base_path) / "token_cache"
TOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

cache_miss = 0
for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc="Tokenizing rows"):
    out_path = TOKEN_CACHE_DIR / row["token_relpath"]
    if out_path.exists():
        continue
    tokens = tokenize_row_with_encodec(row)
    np.save(out_path, tokens.numpy().astype(np.int16))
    cache_miss += 1

print("Token cache dir:", TOKEN_CACHE_DIR)
print("New token files created:", cache_miss)
print("Total token files:", len(list(TOKEN_CACHE_DIR.glob("*.npy"))))

Tokenizing rows:   0%|          | 0/325 [00:00<?, ?it/s]

Token cache dir: /content/drive/MyDrive/data_minecapmf/data/processed/token_cache
New token files created: 0
Total token files: 565


In [ ]:
# --- Quick token/decode sanity check ---
RUN_DIR = Path(base_path) / "runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)

sample_row = split_df.iloc[0]
sample_tokens = np.load(TOKEN_CACHE_DIR / sample_row["token_relpath"])
print("sample file:", sample_row["file_name"])
print("sample split:", sample_row["split"], "aug:", sample_row["virtual_aug"])
print("token shape:", sample_tokens.shape, "min:", sample_tokens.min(), "max:", sample_tokens.max())

recon = decode_tokens_with_encodec(torch.tensor(sample_tokens, dtype=torch.long))
recon_path = RUN_DIR / "sanity_reconstruction.wav"
sf.write(str(recon_path), recon, ENCODEC_SR)
print("Saved reconstruction:", recon_path)

sample file: ambient/cave/cave1_slow.wav
sample split: train aug: none
token shape: (600,) min: 1 max: 1022
Saved reconstruction: /content/drive/MyDrive/data_minecapmf/data/processed/runs/sanity_reconstruction.wav


## Step 3: Small Text-Conditioned Transformer (from scratch)

Design choices (aligned with your plan):
- Decoder-only causal transformer with cross-attention to text encoder memory.
- Frozen text encoder: `t5-small` encoder.
- Token sequence length: `600` (4s, 1.5kbps, 2 codebooks interleaved).
- Codebook vocab: `1024`, plus BOS token.

In [ ]:
# --- Data pipeline for training ---
from transformers import AutoTokenizer, T5EncoderModel, get_cosine_schedule_with_warmup

BOS_ID = CODEBOOK_SIZE
VOCAB_SIZE = CODEBOOK_SIZE + 1  # only BOS extra token
TEXT_MODEL_NAME = "t5-small"
TEXT_MAX_LEN = 32

# Hyperparameters (from your spec; adjust by hardware)
CFG = {
    "layers": 10,
    "d_model": 512,
    "n_heads": 8,
    "ffn_dim": 2048,
    "dropout": 0.1,
    "lr": 3e-4,
    "betas": (0.9, 0.95),
    "weight_decay": 0.1,
    "batch_size": 16 if DEVICE == "cuda" else 2,
    "epochs": 50,
    "grad_accum_steps": 1,
    "warmup_ratio": 0.05,
    "min_lr": 1e-5,
    "max_grad_norm": 1.0,
    "eval_every_steps": 100,
}

print(CFG)

class TokenCaptionDataset(Dataset):
    def __init__(self, df_split: pd.DataFrame, token_cache_dir: Path):
        self.df = df_split.reset_index(drop=True)
        self.token_cache_dir = token_cache_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        toks = np.load(self.token_cache_dir / row["token_relpath"]).astype(np.int64)
        toks = torch.from_numpy(toks)
        return {
            "tokens": toks,
            "caption": row["caption"],
            "event_key": row["event_key"],
            "file_name": row["file_name"],
        }


def collate_fn(batch):
    tokens = torch.stack([b["tokens"] for b in batch], dim=0)  # [B, L]
    captions = [b["caption"] for b in batch]
    return {
        "tokens": tokens,
        "captions": captions,
    }

train_df = split_df[split_df["split"] == "train"].copy()
val_df = split_df[split_df["split"] == "val"].copy()
test_df = split_df[split_df["split"] == "test"].copy()

train_ds = TokenCaptionDataset(train_df, TOKEN_CACHE_DIR)
val_ds = TokenCaptionDataset(val_df, TOKEN_CACHE_DIR)
test_ds = TokenCaptionDataset(test_df, TOKEN_CACHE_DIR)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_fn)

print("train/val/test rows:", len(train_ds), len(val_ds), len(test_ds))
print("steps/epoch:", math.ceil(len(train_loader) / CFG["grad_accum_steps"]))

text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = T5EncoderModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
text_encoder.eval()
for p in text_encoder.parameters():
    p.requires_grad = False

print("Loaded frozen text encoder:", TEXT_MODEL_NAME)


{'layers': 10, 'd_model': 512, 'n_heads': 8, 'ffn_dim': 2048, 'dropout': 0.1, 'lr': 0.0003, 'betas': (0.9, 0.95), 'weight_decay': 0.1, 'batch_size': 16, 'epochs': 50, 'grad_accum_steps': 1, 'warmup_ratio': 0.05, 'min_lr': 1e-05, 'max_grad_norm': 1.0, 'eval_every_steps': 100}
train/val/test rows: 154 106 65
steps/epoch: 10


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

Loaded frozen text encoder: t5-small


In [ ]:
# --- Small causal decoder with cross-attention to text memory ---
class TextCondTokenTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_heads: int,
        num_layers: int,
        ffn_dim: int,
        dropout: float,
        max_seq_len: int,
        n_codebooks: int,
        text_dim: int,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.n_codebooks = n_codebooks

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.codebook_emb = nn.Embedding(n_codebooks, d_model)

        self.text_proj = nn.Linear(text_dim, d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, text_hidden, text_padding_mask=None):
        # input_ids: [B, T], text_hidden: [B, S, text_dim]
        B, T = input_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"input length {T} > max_seq_len {self.max_seq_len}")

        pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
        cb = (torch.arange(T, device=input_ids.device) % self.n_codebooks).unsqueeze(0).expand(B, T)

        x = self.token_emb(input_ids) + self.pos_emb(pos) + self.codebook_emb(cb)
        mem = self.text_proj(text_hidden)

        causal_mask = torch.triu(
            torch.full((T, T), float("-inf"), device=input_ids.device),
            diagonal=1,
        )

        h = self.decoder(
            tgt=x,
            memory=mem,
            tgt_mask=causal_mask,
            memory_key_padding_mask=text_padding_mask,
        )
        h = self.norm(h)
        logits = self.lm_head(h)
        return logits


model = TextCondTokenTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=CFG["d_model"],
    n_heads=CFG["n_heads"],
    num_layers=CFG["layers"],
    ffn_dim=CFG["ffn_dim"],
    dropout=CFG["dropout"],
    max_seq_len=TOKEN_SEQ_LEN,
    n_codebooks=N_CODEBOOKS,
    text_dim=text_encoder.config.d_model,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {num_params/1e6:.2f}M")


Model params: 43.66M


In [ ]:
# --- Training utilities ---
def encode_text_batch(captions):
    tok = text_tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=TEXT_MAX_LEN,
        return_tensors="pt",
    )
    tok = {k: v.to(DEVICE) for k, v in tok.items()}
    with torch.no_grad():
        out = text_encoder(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    text_hidden = out.last_hidden_state
    text_padding_mask = tok["attention_mask"] == 0  # True means ignore
    return text_hidden, text_padding_mask


def batch_to_inputs_targets(tokens):
    # tokens: [B, L] with values in [0, 1023]
    B, L = tokens.shape
    bos = torch.full((B, 1), BOS_ID, dtype=torch.long, device=tokens.device)
    inp = torch.cat([bos, tokens[:, :-1]], dim=1)
    tgt = tokens
    return inp, tgt


def evaluate_loss(model, loader, max_batches=None):
    model.eval()
    losses = []
    with torch.no_grad():
        for bi, batch in enumerate(loader):
            if max_batches is not None and bi >= max_batches:
                break
            tokens = batch["tokens"].to(DEVICE)
            text_hidden, text_padding_mask = encode_text_batch(batch["captions"])
            inp, tgt = batch_to_inputs_targets(tokens)
            logits = model(inp, text_hidden, text_padding_mask=text_padding_mask)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
            losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")


In [ ]:
# --- Train loop ---
optimizer = AdamW(
    model.parameters(),
    lr=CFG["lr"],
    betas=CFG["betas"],
    weight_decay=CFG["weight_decay"],
)

steps_per_epoch = math.ceil(len(train_loader) / CFG["grad_accum_steps"])
total_steps = CFG["epochs"] * steps_per_epoch
warmup_steps = int(CFG["warmup_ratio"] * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print("total_steps:", total_steps, "warmup_steps:", warmup_steps)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

global_step = 0
best_val = float("inf")
history = []

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    running = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CFG['epochs']}")
    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(pbar, start=1):
        tokens = batch["tokens"].to(DEVICE)
        text_hidden, text_padding_mask = encode_text_batch(batch["captions"])
        inp, tgt = batch_to_inputs_targets(tokens)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda"), dtype=torch.float16):
            logits = model(inp, text_hidden, text_padding_mask=text_padding_mask)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
            loss = loss / CFG["grad_accum_steps"]

        scaler.scale(loss).backward()

        if i % CFG["grad_accum_steps"] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            step_loss = float(loss.item() * CFG["grad_accum_steps"])
            running.append(step_loss)
            pbar.set_postfix({"loss": f"{np.mean(running[-20:]):.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

            if global_step % CFG["eval_every_steps"] == 0:
                val_loss = evaluate_loss(model, val_loader, max_batches=None)
                train_loss = float(np.mean(running[-100:])) if running else float("nan")
                history.append({
                    "step": global_step,
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "lr": scheduler.get_last_lr()[0],
                })
                print(f"\nstep={global_step} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

                if val_loss < best_val:
                    best_val = val_loss
                    ckpt = {
                        "model_state": model.state_dict(),
                        "config": CFG,
                        "best_val": best_val,
                        "global_step": global_step,
                        "token_seq_len": TOKEN_SEQ_LEN,
                        "vocab_size": VOCAB_SIZE,
                        "bos_id": BOS_ID,
                        "n_codebooks": N_CODEBOOKS,
                    }
                    torch.save(ckpt, RUN_DIR / "best_model.pt")
                    print("Saved best checkpoint ->", RUN_DIR / "best_model.pt")

# save final artifacts
hist_df = pd.DataFrame(history)
hist_path = RUN_DIR / "train_history.csv"
hist_df.to_csv(hist_path, index=False)
torch.save({"model_state": model.state_dict(), "config": CFG, "global_step": global_step}, RUN_DIR / "last_model.pt")
print("Training complete.")
print("Best val:", best_val)
print("History:", hist_path)
print("Final checkpoint:", RUN_DIR / "last_model.pt")


total_steps: 500 warmup_steps: 25


/tmp/ipykernel_14898/2288218415.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


Epoch 1/50:   0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_14898/2288218415.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda"), dtype=torch.float16):
/tmp/ipykernel_14898/2288218415.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda"), dtype=torch.float16):


Epoch 2/50:   0%|          | 0/10 [00:00<?, ?it/s]

...

Epoch 47/50:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 48/50:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/10 [00:00<?, ?it/s]


step=500 train_loss=1.9896 val_loss=4.2012
Training complete.
Best val: 4.098590646471296
History: /content/drive/MyDrive/data_minecapmf/data/processed/runs/train_history.csv
Final checkpoint: /content/drive/MyDrive/data_minecapmf/data/processed/runs/last_model.pt


## Step 4: Inference (Text -> Tokens -> Audio)

Sampling uses top-k + temperature as requested.

In [ ]:
# --- Sampling and waveform generation ---
def sample_from_logits(logits, temperature=0.9, top_k=250):
    # logits: [B, V]
    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    logits = logits / temperature

    # Only sample actual EnCodec token ids [0..1023], never BOS
    logits = logits[:, :CODEBOOK_SIZE]

    if top_k is not None and top_k > 0:
        k = min(top_k, logits.shape[-1])
        vals, idx = torch.topk(logits, k=k, dim=-1)
        probs = torch.softmax(vals, dim=-1)
        next_local = torch.multinomial(probs, num_samples=1)
        next_token = idx.gather(-1, next_local)
    else:
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

    return next_token


def generate_token_sequence(prompt, max_tokens=TOKEN_SEQ_LEN, temperature=0.9, top_k=250):
    model.eval()

    text_hidden, text_padding_mask = encode_text_batch([prompt])

    seq = torch.tensor([[BOS_ID]], dtype=torch.long, device=DEVICE)  # [1,1]
    for _ in tqdm(range(max_tokens), desc="Generating tokens"):
        inp = seq[:, -TOKEN_SEQ_LEN:]
        logits = model(inp, text_hidden, text_padding_mask=text_padding_mask)
        next_token = sample_from_logits(logits[:, -1, :], temperature=temperature, top_k=top_k)
        seq = torch.cat([seq, next_token], dim=1)

    out_tokens = seq[:, 1:1 + max_tokens].squeeze(0).detach().cpu()
    return out_tokens


def save_generated_audio(prompt, out_wav_path, temperature=0.9, top_k=250):
    tokens = generate_token_sequence(prompt, max_tokens=TOKEN_SEQ_LEN, temperature=temperature, top_k=top_k)
    wav = decode_tokens_with_encodec(tokens)
    sf.write(str(out_wav_path), wav, ENCODEC_SR)
    return out_wav_path


PROMPT = "minecraft zombie groaning in a cave with short footsteps"
out_wav = RUN_DIR / "gen_zombie_cave.wav"
path = save_generated_audio(PROMPT, out_wav, temperature=0.9, top_k=250)
print("Saved generated wav:", path)


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/data/processed/runs/gen_zombie_cave.wav


In [ ]:
from IPython.display import Audio

# Define a list of example captions/prompts
example_prompts = [
    "minecraft zombie groaning in a cave with short footsteps",
    "minecraft cave ambience sound effect",
    "minecraft player taking damage",
    "minecraft skeleton walking quickly footsteps sound effect",
    "minecraft water splash exiting sound effect",
    "fast minecraft blaze breathing sound effect"
]

print("--- Generating Multiple Audios ---")
generated_audios_data = []
for i, prompt in enumerate(example_prompts):
    output_filename = RUN_DIR / f"gen_example_audio_{i+1}.wav"
    print(f"\nGenerating audio for prompt: \"{prompt}\"")
    path = save_generated_audio(prompt, output_filename, temperature=0.9, top_k=250)
    generated_audios_data.append((prompt, path))
    print(f"Saved generated wav: {path}")

# Display all generated audios
print("\n--- Displaying Generated Audios ---")
for prompt, audio_path in generated_audios_data:
    print(f"\nPrompt: \"{prompt}\"")
    y_gen, sr_gen = librosa.load(audio_path, sr=None)
    display(Audio(y_gen, rate=sr_gen))

# Display one example ground truth audio for comparison
# Find a suitable ground truth audio (e.g., a zombie sound)
zombie_ground_truth_row = split_df[
    (split_df['caption'].str.contains('zombie', case=False)) &
    (split_df['is_virtual'] == 0)
].iloc[0]

ground_truth_file_name = zombie_ground_truth_row['file_name']
ground_truth_caption = zombie_ground_truth_row['caption']
ground_truth_audio_path = AUDIO_ROOT / ground_truth_file_name

print(f"\n--- Ground Truth Audio (Example) ---")
print(f"Ground Truth Audio: {ground_truth_audio_path}")
print(f"Ground Truth Caption: {ground_truth_caption}")
y_gt, sr_gt = librosa.load(ground_truth_audio_path, sr=None)
display(Audio(y_gt, rate=sr_gt))

--- Generating Multiple Audios ---

Generating audio for prompt: "minecraft zombie groaning in a cave with short footsteps"


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/data/processed/runs/gen_example_audio_1.wav

Generating audio for prompt: "minecraft cave ambience sound effect"


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/data/processed/runs/gen_example_audio_2.wav

Generating audio for prompt: "minecraft player taking damage"


...


Prompt: "minecraft player taking damage"



Prompt: "minecraft skeleton walking quickly footsteps sound effect"



Prompt: "minecraft water splash exiting sound effect"



Prompt: "fast minecraft blaze breathing sound effect"



--- Ground Truth Audio (Example) ---
Ground Truth Audio: /content/drive/MyDrive/data_minecapmf/data/processed/combat/damage_zombie_death.wav
Ground Truth Caption: minecraft player taking damage and zombie dying sound effect
